## 1. Import Libraries dan Setup

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
from clickhouse_connect import get_client
import re

# Konfigurasi
TABLE_NAME = "dataset.anggaran"
EXCEL_FILE = "/Users/rezkyyayang/Documents/Code/Realisasi 31 Okt 2026.xlsx"
TAHUN = "2026"
TANGGAL_REALISASI = "2026-10-31"  # Format YYYY-MM-DD

# Koneksi ke ClickHouse
client = get_client(
    host=,
    port=,
    username=,
    password=,
    database=
)

print("✓ Connected to ClickHouse")
print(f"✓ Table: {TABLE_NAME}")

✓ Connected to ClickHouse
✓ Table: dataset.anggaran


## 2. Fungsi Helper dan Parser

In [ ]:
def get_next_id(client):
    """Get next available ID from database"""
    result = client.query(f"SELECT max(id) FROM {TABLE_NAME}")
    max_id = result.result_rows[0][0]
    return (max_id or 0) + 1

def clean_number(value):
    """Convert number string to clean number (remove commas, decimals, etc)"""
    if pd.isna(value):
        return None

    value = str(value).strip()
    if not value or value.lower() in ['nan', 'none', '']:
        return None

    # Remove formatting characters
    value = value.replace('.', '').replace(',', '')

    try:
        num = float(value)
        # Return as string to preserve precision
        return str(int(num)) if num == int(num) else str(num)
    except:
        return None

def format_percentage(value):
    """Convert percentage string to float"""
    if pd.isna(value):
        return None

    try:
        val = float(str(value).strip())
        return round(val, 6)  # 6 decimal places for percentage
    except:
        return None

def clean_satker_name(value):
    """Clean and normalize satker name"""
    if pd.isna(value):
        return None

    value = str(value).strip()
    if not value or value.lower() in ['nan', 'none', '']:
        return None

    # Remove trailing pipes and asterisks
    value = re.sub(r'[|*]\s*$', '', value).strip()
    return value if value else None

print("✓ Helper functions loaded")

✓ Helper functions loaded


## 3. Parse Sheet: Per SumberDana (By Funding Source)

In [ ]:
def parse_per_sumber_dana_v2(excel_file, tahun_date=TANGGAL_REALISASI):
    """
    Parse 'per SumberDana' sheet - By Funding Source

    Excel columns (0-indexed):
    - Col 0: No (satker number)
    - Col 1: NAMA SATKER / klasifikasi
    - Col 2: PAGU * (pagu_anggaran) ← USE THIS
    - Col 3: PAGU PEMANFAATAN / STLH BLOKIR (ignore this)
    - Col 4: REALISASI (sp2d_span) ← USE THIS
    - Col 5: SISA PAGU (sisa_dana) ← USE THIS
    - Col 6: % SP2D (persen_realisasi) ← USE THIS

    Note: Only include records with non-empty klasifikasi (skip satker total rows)
    """
    df_raw = pd.read_excel(excel_file, sheet_name="realisasi per sumber dana- SP2D", header=None)
    records = []

    for idx, row in df_raw.iterrows():
        col0 = str(row.iloc[0]).strip() if pd.notna(row.iloc[0]) else ""
        col1 = str(row.iloc[1]).strip() if pd.notna(row.iloc[1]) else ""
        col2 = row.iloc[2] if pd.notna(row.iloc[2]) else None  # PAGU *
        col4 = row.iloc[3] if pd.notna(row.iloc[3]) else None  # REALISASI
        col5 = row.iloc[4] if pd.notna(row.iloc[4]) else None  # SISA PAGU
        col6 = row.iloc[5] if pd.notna(row.iloc[5]) else None  # % SP2D

        # Only process rows with klasifikasi (pipe separator) - skip satker total rows
        if "|" in col1:
            klasifikasi = col1
            pagu = clean_number(col2)
            realisasi = clean_number(col4)
            sisa = clean_number(col5)
            persen = format_percentage(col6)

            if col0.isdigit():
                # Get satker name from previous row if available
                if idx > 0:
                    prev_row = df_raw.iloc[idx-1]
                    prev_col1 = str(prev_row.iloc[1]).strip() if pd.notna(prev_row.iloc[1]) else ""
                    if prev_col1 and "|" not in prev_col1:
                        satker_lan = prev_col1
                    else:
                        satker_lan = None
                else:
                    satker_lan = None
            else:
                # Find satker name from above
                satker_lan = None
                for prev_idx in range(idx-1, -1, -1):
                    prev_row = df_raw.iloc[prev_idx]
                    prev_col0 = str(prev_row.iloc[0]).strip() if pd.notna(prev_row.iloc[0]) else ""
                    prev_col1 = str(prev_row.iloc[1]).strip() if pd.notna(prev_row.iloc[1]) else ""
                    if prev_col0.isdigit() and prev_col1 and "|" not in prev_col1:
                        satker_lan = prev_col1
                        break

            if satker_lan and pagu:
                records.append({
                    'satker_lan': satker_lan,
                    'tahun': tahun_date,
                    'disagregasi': 'SUMBER DANA',
                    'klasifikasi': klasifikasi,
                    'alias': '',
                    'pagu_anggaran': pagu,
                    'sp2d_span': realisasi or '',
                    'sisa_dana': sisa or '',
                    'persen_realisasi': persen or 0.0
                })

    result = pd.DataFrame(records)
    print(f"✓ Parsed 'per SumberDana': {len(result)} records")
    if len(result) > 0:
        print(f"  Klasifikasi types: {sorted(result['klasifikasi'].unique().tolist())}")
        print(f"  Persen range: {result['persen_realisasi'].min():.2f}% - {result['persen_realisasi'].max():.2f}%")
    return result


# Test
print("\n" + "="*80)
print("PARSING: Per Sumber Dana (Fixed)")
print("="*80)

df_sumber_dana = parse_per_sumber_dana_v2(EXCEL_FILE, tahun_date=TANGGAL_REALISASI)
print("\nSample output:")
print(df_sumber_dana[['satker_lan', 'disagregasi', 'klasifikasi', 'pagu_anggaran', 'sp2d_span', 'persen_realisasi']].head(15).to_string(index=False))


PARSING: Per Sumber Dana (Fixed)


✓ Parsed 'per SumberDana': 17 records
  Klasifikasi types: ['A | RUPIAH MURNI', 'D | PENERIMAAN NEGARA BUKAN PAJAK']
  Persen range: 43.20% - 50172763695.00%

Sample output:
                  satker_lan disagregasi                       klasifikasi pagu_anggaran   sp2d_span  persen_realisasi
                 LAN JAKARTA SUMBER DANA                  A | RUPIAH MURNI  116401904000 91881326289         78.934556
                 LAN JAKARTA SUMBER DANA D | PENERIMAAN NEGARA BUKAN PAJAK   14697739000 10111344352         68.795237
 POLITEKNIK STIA LAN Jakarta SUMBER DANA                  A | RUPIAH MURNI   18170798000 14156526866         77.908119
 POLITEKNIK STIA LAN Jakarta SUMBER DANA D | PENERIMAAN NEGARA BUKAN PAJAK   13362975000  5773204894         43.202991
 POLITEKNIK STIA LAN Bandung SUMBER DANA                  A | RUPIAH MURNI   19566136000 15272306679         78.054792
 POLITEKNIK STIA LAN Bandung SUMBER DANA D | PENERIMAAN NEGARA BUKAN PAJAK   10642410000  7168059985         67.

### 3.2 Parser: Per Jenis Belanja

In [ ]:
def parse_per_jenis_belanja_v2(excel_file, tahun_date=TANGGAL_REALISASI):
    """
    Parse 'per jenis belanja' sheet - By Expense Type

    Excel columns (0-indexed):
    - Col 0: No (satker number)
    - Col 1: NAMA SATKER / klasifikasi
    - Col 2: PAGU * (pagu_anggaran) ← USE THIS
    - Col 3: PAGU PEMANFAATAN / STLH BLOKIR (ignore this)
    - Col 4: REALISASI (sp2d_span) ← USE THIS
    - Col 5: SISA PAGU (sisa_dana) ← USE THIS
    - Col 6: % SP2D (persen_realisasi) ← USE THIS

    Note: Only include records with non-empty klasifikasi (skip satker total rows)
    """
    df_raw = pd.read_excel(excel_file, sheet_name="realisasi per jenis blnj SP2D", header=None)
    records = []

    for idx, row in df_raw.iterrows():
        col0 = str(row.iloc[0]).strip() if pd.notna(row.iloc[0]) else ""
        col1 = str(row.iloc[1]).strip() if pd.notna(row.iloc[1]) else ""
        col2 = row.iloc[2] if pd.notna(row.iloc[2]) else None  # PAGU *
        col4 = row.iloc[3] if pd.notna(row.iloc[3]) else None  # REALISASI
        col5 = row.iloc[4] if pd.notna(row.iloc[4]) else None  # SISA PAGU
        col6 = row.iloc[5] if pd.notna(row.iloc[5]) else None  # % SP2D

        # Only process rows with klasifikasi (pipe separator) - skip satker total rows
        if "|" in col1:
            klasifikasi = col1
            pagu = clean_number(col2)
            realisasi = clean_number(col4)
            sisa = clean_number(col5)
            persen = format_percentage(col6)

            if col0.isdigit():
                # Get satker name from previous row if available
                if idx > 0:
                    prev_row = df_raw.iloc[idx-1]
                    prev_col1 = str(prev_row.iloc[1]).strip() if pd.notna(prev_row.iloc[1]) else ""
                    if prev_col1 and "|" not in prev_col1:
                        satker_lan = prev_col1
                    else:
                        satker_lan = None
                else:
                    satker_lan = None
            else:
                # Find satker name from above
                satker_lan = None
                for prev_idx in range(idx-1, -1, -1):
                    prev_row = df_raw.iloc[prev_idx]
                    prev_col0 = str(prev_row.iloc[0]).strip() if pd.notna(prev_row.iloc[0]) else ""
                    prev_col1 = str(prev_row.iloc[1]).strip() if pd.notna(prev_row.iloc[1]) else ""
                    if prev_col0.isdigit() and prev_col1 and "|" not in prev_col1:
                        satker_lan = prev_col1
                        break

            if satker_lan and pagu:
                records.append({
                    'satker_lan': satker_lan,
                    'tahun': tahun_date,
                    'disagregasi': 'JENIS BELANJA',
                    'klasifikasi': klasifikasi,
                    'alias': '',
                    'pagu_anggaran': pagu,
                    'sp2d_span': realisasi or '',
                    'sisa_dana': sisa or '',
                    'persen_realisasi': persen or 0.0
                })

    result = pd.DataFrame(records)
    print(f"✓ Parsed 'per jenis belanja': {len(result)} records")
    if len(result) > 0:
        print(f"  Klasifikasi types: {sorted(result['klasifikasi'].unique().tolist())}")
        print(f"  Persen range: {result['persen_realisasi'].min():.2f}% - {result['persen_realisasi'].max():.2f}%")
    return result


# Test
print("\n" + "="*80)
print("PARSING: Per Jenis Belanja (Fixed)")
print("="*80)

df_jenis_belanja = parse_per_jenis_belanja_v2(EXCEL_FILE, tahun_date=TANGGAL_REALISASI)
print("\nSample output:")
print(df_jenis_belanja[['satker_lan', 'disagregasi', 'klasifikasi', 'pagu_anggaran', 'sp2d_span', 'persen_realisasi']].head(20).to_string(index=False))


PARSING: Per Jenis Belanja (Fixed)
✓ Parsed 'per jenis belanja': 25 records
  Klasifikasi types: ['51 | BELANJA PEGAWAI', '52 | BELANJA BARANG', '53 | BELANJA MODAL']
  Persen range: 9.72% - 32419885005.00%

Sample output:
                  satker_lan   disagregasi          klasifikasi pagu_anggaran   sp2d_span  persen_realisasi
                 LAN JAKARTA JENIS BELANJA 51 | BELANJA PEGAWAI   78582329000 63784058958         81.168451
                 LAN JAKARTA JENIS BELANJA  52 | BELANJA BARANG   51502055000 38109928183         73.996908
                 LAN JAKARTA JENIS BELANJA   53 | BELANJA MODAL    1015259000    98683500          9.720032
 POLITEKNIK STIA LAN Jakarta JENIS BELANJA 51 | BELANJA PEGAWAI   14913878000 11868009368         79.576951
 POLITEKNIK STIA LAN Jakarta JENIS BELANJA  52 | BELANJA BARANG   13306279000  7519019058         56.507300
 POLITEKNIK STIA LAN Jakarta JENIS BELANJA   53 | BELANJA MODAL    3313616000   542703334         16.377979
 POLITEKNIK STIA LAN

### 4. Parse per Unit Kerja

In [ ]:
def parse_realisasi_sp2d_lan_jkt_eselon_ii(excel_file, tahun_date=TANGGAL_REALISASI):
    """
    Parse 'realisasi sp2d lan jkt' sheet - Extract ESELON II units (Biro/Direktorat)

    Data structure:
    - Main eselon II rows: Col0 has single letter (A,B,C,D,E), Col1 has full org name
    - Each main row has pagu, sp2d, sisa, persen data in cols 2-5
    - Col19 may have short org names/abbreviations
    """
    df_raw = pd.read_excel(excel_file, sheet_name="realisasi bend lan jkt", header=None)
    records = []

    for idx, row in df_raw.iterrows():
        col0 = str(row.iloc[0]).strip() if pd.notna(row.iloc[0]) else ""
        col1 = str(row.iloc[1]).strip() if pd.notna(row.iloc[1]) else ""
        col2 = row.iloc[2] if pd.notna(row.iloc[2]) else None
        col3 = row.iloc[3] if pd.notna(row.iloc[3]) else None
        col4 = row.iloc[4] if pd.notna(row.iloc[4]) else None
        col5 = row.iloc[5] if pd.notna(row.iloc[5]) else None
        col19 = str(row.iloc[19]).strip() if len(row) > 19 and pd.notna(row.iloc[19]) else ""

        # Check if this is a main ESELON II row:
        # Col0 is single letter (A,B,C,D,E) and Col1 starts with "Biro" or "Deputi" or "Direktorat" (main org level)
        # AND has numeric pagu value > 0
        is_main_org = (
            len(col0) == 1 and col0.isalpha() and
            (col1.startswith('Biro ') or col1.startswith('Deputi ') or col1.startswith('Direktorat ') or col1.startswith('Inspektorat') or col1.startswith('Pusat Data'))
        )
        has_pagu = pd.notna(col2) and isinstance(col2, (int, float)) and col2 > 0

        if is_main_org and has_pagu:
            klasifikasi = col1.replace('\n', ' ').strip()
            alias = col19 or ''

            pagu = clean_number(col2)
            sp2d = clean_number(col3)
            sisa = clean_number(col4)
            persen_str = col5 if pd.notna(col5) else 0
            persen = format_percentage(persen_str)

            if pagu:
                records.append({
                    'satker_lan': 'LAN JAKARTA',
                    'tahun': tahun_date,
                    'disagregasi': 'ESELON II',
                    'klasifikasi': klasifikasi,
                    'alias': alias,
                    'pagu_anggaran': pagu,
                    'sp2d_span': sp2d or '',
                    'sisa_dana': sisa or '',
                    'persen_realisasi': persen or 0.0
                })

    result = pd.DataFrame(records)
    print(f"✓ Parsed ESELON II: {len(result)} records")
    if len(result) > 0:
        klasifikasi_list = result['klasifikasi'].unique().tolist()
        print(f"  Units: {klasifikasi_list}")
    return result


# Test
print("\n" + "="*80)
print("PARSING: Realisasi SP2D LAN JKT (ESELON II)")
print("="*80)

df_eselon_ii = parse_realisasi_sp2d_lan_jkt_eselon_ii(EXCEL_FILE, tahun_date=TANGGAL_REALISASI)
print("\nSample output:")
cols_to_show = ['satker_lan', 'disagregasi', 'klasifikasi', 'alias', 'pagu_anggaran', 'persen_realisasi']
print(df_eselon_ii[cols_to_show].to_string(index=False))


PARSING: Realisasi SP2D LAN JKT (ESELON II)
✓ Parsed ESELON II: 16 records
  Units: ['Biro Perencanaan dan Keuangan', 'Biro Hukum, Organisasi, dan Sumber Daya Manusia', 'Biro Umum, Kerja Sama, dan Hubungan Masyarakat', 'Inspektorat', 'Pusat Data dan Informasi', 'Direktorat Strategi Peningkatan Kualitas Kebijakan Administrasi Negara', 'Direktorat Advokasi dan Pengembangan Kinerja Kebijakan', 'Direktorat Penguatan Kapasitas Jabatan Fungsional Bidang Pengembangan Kapasitas dan Pembelajaran ASN', 'Direktorat Sistem Pembelajaran Terintegrasi', 'Direktorat Teknologi dan Digitalisasi Pembelajaran', 'Direktorat Ekosistem Pembelajaran Aparatur Sipil Negara', 'Direktorat Pembelajaran Manajerial Kepemimpinan', 'Direktorat Pembelajaran Karakter dan Sosial Kultural', 'Direktorat Pembelajaran Teknis dan Fungsional', 'Direktorat Penjaminan Mutu Pengembangan Kapasitas', 'Direktorat Penjaminan Mutu Pembelajaran']

Sample output:
 satker_lan disagregasi                                                  

### 4. Combine All Data Sources

In [ ]:
# Combine all data sources
print("\n" + "="*80)
print("COMBINING DATA")
print("="*80)

# Parse all sheets with updated parsers
df_sumber_dana_new = parse_per_sumber_dana_v2(EXCEL_FILE, tahun_date=TANGGAL_REALISASI)
df_jenis_belanja_new = parse_per_jenis_belanja_v2(EXCEL_FILE, tahun_date=TANGGAL_REALISASI)
df_eselon_ii_new = parse_realisasi_sp2d_lan_jkt_eselon_ii(EXCEL_FILE, tahun_date=TANGGAL_REALISASI)

# Combine all dataframes
df_combined = pd.concat([
    df_sumber_dana_new,
    df_jenis_belanja_new,
    df_eselon_ii_new
], ignore_index=True)

print(f"\n✓ Combined data:")
print(f"  - Sumber Dana: {len(df_sumber_dana_new)} records")
print(f"  - Jenis Belanja: {len(df_jenis_belanja_new)} records")
print(f"  - Eselon II: {len(df_eselon_ii_new)} records")
print(f"  - Total: {len(df_combined)} records")

print(f"\nBy disagregasi:")
print(df_combined['disagregasi'].value_counts().to_string())

print(f"\nData ready for insertion:")
print(df_combined[['satker_lan', 'disagregasi', 'klasifikasi', 'pagu_anggaran']].head(10).to_string(index=False))


COMBINING DATA
✓ Parsed 'per SumberDana': 17 records
  Klasifikasi types: ['A | RUPIAH MURNI', 'D | PENERIMAAN NEGARA BUKAN PAJAK']
  Persen range: 43.20% - 50172763695.00%
✓ Parsed 'per jenis belanja': 25 records
  Klasifikasi types: ['51 | BELANJA PEGAWAI', '52 | BELANJA BARANG', '53 | BELANJA MODAL']
  Persen range: 9.72% - 32419885005.00%
✓ Parsed ESELON II: 16 records
  Units: ['Biro Perencanaan dan Keuangan', 'Biro Hukum, Organisasi, dan Sumber Daya Manusia', 'Biro Umum, Kerja Sama, dan Hubungan Masyarakat', 'Inspektorat', 'Pusat Data dan Informasi', 'Direktorat Strategi Peningkatan Kualitas Kebijakan Administrasi Negara', 'Direktorat Advokasi dan Pengembangan Kinerja Kebijakan', 'Direktorat Penguatan Kapasitas Jabatan Fungsional Bidang Pengembangan Kapasitas dan Pembelajaran ASN', 'Direktorat Sistem Pembelajaran Terintegrasi', 'Direktorat Teknologi dan Digitalisasi Pembelajaran', 'Direktorat Ekosistem Pembelajaran Aparatur Sipil Negara', 'Direktorat Pembelajaran Manajerial Kepe

### 5. Assign IDs and Insert into ClickHouse

In [ ]:
# Get current max ID from database
print("\n" + "="*80)
print("PREPARING FOR INSERTION")
print("="*80)

# Get total rows before insertion
total_rows_before = client.query(f"SELECT count(*) FROM {TABLE_NAME}").result_rows[0][0]
print(f"Total rows before insertion: {total_rows_before}")

max_id_query = "SELECT max(id) as max_id FROM dataset.anggaran"
max_id_result = client.query(max_id_query)
current_max_id = max_id_result.result_rows[0][0] if max_id_result.result_rows else 0

print(f"Current max ID in database: {current_max_id}")

# Assign new IDs starting from max_id + 1
df_combined['id'] = range(current_max_id + 1, current_max_id + 1 + len(df_combined))

print(f"\nAssigned IDs: {df_combined['id'].min()} to {df_combined['id'].max()}")
print(f"Total records to insert: {len(df_combined)}")

# Prepare data for insertion
insert_data = []
for idx, row in df_combined.iterrows():
    insert_data.append([
        row['satker_lan'],
        row['tahun'],
        row['disagregasi'],
        row['klasifikasi'],
        row['alias'],
        row['pagu_anggaran'],
        row['sp2d_span'],
        row['sisa_dana'],
        row['persen_realisasi'],
        row['id']
    ])

print("\nFirst 3 rows to insert:")
for i in range(min(3, len(insert_data))):
    print(f"  Row {i+1}: {insert_data[i]}")


PREPARING FOR INSERTION
Total rows before insertion: 487
Current max ID in database: 487

Assigned IDs: 488 to 545
Total records to insert: 58

First 3 rows to insert:
  Row 1: ['LAN JAKARTA', '2026-10-31', 'SUMBER DANA', 'A | RUPIAH MURNI', '', '116401904000', '91881326289', '24520577711', 78.934556, 488]
  Row 2: ['LAN JAKARTA', '2026-10-31', 'SUMBER DANA', 'D | PENERIMAAN NEGARA BUKAN PAJAK', '', '14697739000', '10111344352', '4586394648', 68.795237, 489]
  Row 3: ['POLITEKNIK STIA LAN Jakarta', '2026-10-31', 'SUMBER DANA', 'A | RUPIAH MURNI', '', '18170798000', '14156526866', '4014271134', 77.908119, 490]


### 5.1 Insert Data into ClickHouse

In [ ]:
# Insert data into ClickHouse
print("\n" + "="*80)
print("INSERTING DATA INTO CLICKHOUSE")
print("="*80)

try:
    # Insert batch
    insert_query = f"""
    INSERT INTO {TABLE_NAME}
    (satker_lan, tahun, disagregasi, klasifikasi, alias, pagu_anggaran, sp2d_span, sisa_dana, persen_realisasi, id)
    VALUES
    """

    # Use client.insert method which is more efficient
    client.insert(
        TABLE_NAME,
        insert_data,
        column_names=['satker_lan', 'tahun', 'disagregasi', 'klasifikasi', 'alias',
                     'pagu_anggaran', 'sp2d_span', 'sisa_dana', 'persen_realisasi', 'id']
    )

    print(f"✓ Successfully inserted {len(insert_data)} records")
    print(f"  ID range: {df_combined['id'].min()} - {df_combined['id'].max()}")

except Exception as e:
    print(f"❌ Insertion failed: {str(e)}")
    raise


INSERTING DATA INTO CLICKHOUSE
✓ Successfully inserted 58 records
  ID range: 488 - 545


### 6. Verify Insertion

In [ ]:
# Verify insertion
print("\n" + "="*80)
print("VERIFICATION")
print("="*80)

# Check total count
verify_count = client.query(f"SELECT count(*) as total FROM {TABLE_NAME}")
total_count = verify_count.result_rows[0][0]
print(f"\n✓ Total records in database: {total_count}")

# Check newly inserted records
verify_new = client.query(
    f"SELECT count(*) as new_count FROM {TABLE_NAME} WHERE id >= {df_combined['id'].min()}"
)
new_count = verify_new.result_rows[0][0]
print(f"✓ Newly inserted records: {new_count}")

# Check by disagregasi
print(f"\n✓ Records by disagregasi:")
disagregasi_query = f"""
SELECT disagregasi, count(*) as cnt
FROM {TABLE_NAME}
WHERE id >= {df_combined['id'].min()}
GROUP BY disagregasi
ORDER BY disagregasi
"""
result = client.query(disagregasi_query)
for row in result.result_rows:
    print(f"  - {row[0]}: {row[1]} records")

# Show sample of inserted records
print(f"\n✓ Sample of inserted records:")
sample_query = f"""
SELECT satker_lan, tahun, disagregasi, klasifikasi, pagu_anggaran, persen_realisasi
FROM {TABLE_NAME}
WHERE id >= {df_combined['id'].min()} AND id < {df_combined['id'].min() + 5}
ORDER BY id
"""
sample_result = client.query(sample_query)
for i, row in enumerate(sample_result.result_rows, 1):
    print(f"  Row {i}: {row[0][:20]:20s} | {row[2]:15s} | {row[3][:30]:30s} | Pagu: {row[4]}")

print(f"\n✅ Insertion completed successfully!")


VERIFICATION

✓ Total records in database: 545
✓ Newly inserted records: 58

✓ Records by disagregasi:
  - ESELON II: 16 records
  - JENIS BELANJA: 25 records
  - SUMBER DANA: 17 records

✓ Sample of inserted records:
  Row 1: LAN JAKARTA          | SUMBER DANA     | A | RUPIAH MURNI               | Pagu: 116401904000
  Row 2: LAN JAKARTA          | SUMBER DANA     | D | PENERIMAAN NEGARA BUKAN PA | Pagu: 14697739000
  Row 3: POLITEKNIK STIA LAN  | SUMBER DANA     | A | RUPIAH MURNI               | Pagu: 18170798000
  Row 4: POLITEKNIK STIA LAN  | SUMBER DANA     | D | PENERIMAAN NEGARA BUKAN PA | Pagu: 13362975000
  Row 5: POLITEKNIK STIA LAN  | SUMBER DANA     | A | RUPIAH MURNI               | Pagu: 19566136000

✅ Insertion completed successfully!


## 10. Verifikasi Data di ClickHouse

In [ ]:
# Verify insertion
count_after = client.query(f"SELECT count() FROM {TABLE_NAME}").result_rows[0][0]
max_id_after = client.query(f"SELECT max(id) FROM {TABLE_NAME}").result_rows[0][0]

print("After insertion:")
print(f"  Total rows: {count_after} (added {count_after - total_rows_before})")
print(f"  Max ID: {max_id_after}")

# Show summary by disagregasi
summary = client.query_df(
    f"""SELECT
       disagregasi,
       count() as count,
       countDistinct(satker_lan) as satker_count
       FROM {TABLE_NAME}
       WHERE tahun = '{TAHUN}'
       GROUP BY disagregasi
       ORDER BY disagregasi"""
)

print(f"\nSummary by disagregasi (Year {TAHUN}):")
print(summary.to_string(index=False))

After insertion:
  Total rows: 374 (added 55)
  Max ID: 520

Summary by disagregasi (Year 2026):
Empty DataFrame
Columns: []
Index: []


## 11. View Latest Records

In [ ]:
# Show latest records
latest = client.query_df(
    f"""SELECT
       id, satker_lan, tahun, disagregasi, klasifikasi,
       pagu_anggaran, sp2d_span, sisa_dana, persen_realisasi
       FROM {TABLE_NAME}
       WHERE tahun = '{TAHUN}'
       ORDER BY id DESC LIMIT 30"""
)

print(f"Latest {min(30, len(latest))} records from Year {TAHUN}:")
print(latest.to_string(index=False))

Latest 0 records from Year 2026:
Empty DataFrame
Columns: []
Index: []
